# Synthetic Testset Generation

Generates question/answer pairs from CVE documents for evaluating the vuln-chat RAG pipeline.

Three query types:
- **Single-hop**: answerable from a single CVE document
- **Multi-hop**: requires reasoning across two CVE documents
- **Global**: asks about patterns across many CVE documents

Saves the testset to `testset.json` for use by `eval.ipynb`.

In [5]:
import os

import httpx
from langchain_openai import ChatOpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "")
CHAT_MODEL = os.getenv("CHAT_MODEL", "")

FAISS_URL = os.getenv("FAISS_URL", "http://localhost:8000")

llm = ChatOpenAI(model=CHAT_MODEL, base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)

### Load CVE documents

In [6]:
docs_resp = httpx.get(f"{FAISS_URL}/documents", timeout=30.0)
docs_resp.raise_for_status()
documents = docs_resp.json()["documents"]
print(f"Loaded {len(documents)} CVE documents")
print(f"Example: {documents[0]['metadata']['cve_id']}")

Loaded 20 CVE documents
Example: CVE-2021-44228


### Generate synthetic tests

In [7]:
import random
from test_generators import TestGenerator, Persona

personas = [
    Persona(
        name="security analyst",
        description="A junior security analyst responsible for triaging incoming vulnerability reports and determining patch priority",
    ),
    Persona(
        name="infrastructure engineer",
        description="A DevOps engineer managing containerized deployments who needs to understand container escape and kernel vulnerabilities",
    ),
    Persona(
        name="product manager",
        description="A non-technical product manager who needs plain-language explanations of vulnerability impact and urgency to make prioritization decisions",
    ),
]

generator = TestGenerator(llm=llm)


def format_doc(doc: dict) -> str:
    m = doc.get("metadata", {})
    return (
        f"CVE ID: {m.get('cve_id', 'N/A')}\n"
        f"Severity: {m.get('severity', 'N/A')} (CVSS {m.get('cvss_score', 'N/A')})\n"
        f"Attack Vector: {m.get('attack_vector', 'N/A')}\n"
        f"CWE: {m.get('cwe', 'N/A')}\n"
        f"Description: {doc['content']}"
    )


def find_multi_hop_pairs(docs: list[dict]) -> list[tuple[dict, dict]]:
    pairs = []
    for i, a in enumerate(docs):
        for b in docs[i + 1:]:
            ma, mb = a.get("metadata", {}), b.get("metadata", {})
            cwe_a = {c.strip() for c in ma.get("cwe", "").split(",") if c.strip()}
            cwe_b = {c.strip() for c in mb.get("cwe", "").split(",") if c.strip()}
            if cwe_a & cwe_b:
                pairs.append((a, b))
                continue
            if ma.get("attack_vector") and ma["attack_vector"] == mb.get("attack_vector"):
                pairs.append((a, b))
    return pairs


synthetic_tests = []

# Single-hop
sample_docs = random.sample(documents, min(5, len(documents)))
for i, doc in enumerate(sample_docs):
    persona = personas[i % len(personas)]
    cve_id = doc["metadata"].get("cve_id", "?")
    print(f"  single-hop {i+1}/{len(sample_docs)}: {cve_id}, {persona.name}")
    test = generator.generate_single_hop(format_doc(doc), persona)
    synthetic_tests.append({"user_input": test.question, "reference": test.answer, "query_type": "single_hop", "persona": persona.name, "source_cves": cve_id})

# Multi-hop
pairs = find_multi_hop_pairs(documents)
sample_pairs = random.sample(pairs, min(3, len(pairs)))
for i, (a, b) in enumerate(sample_pairs):
    persona = personas[i % len(personas)]
    id_a, id_b = a["metadata"].get("cve_id", "?"), b["metadata"].get("cve_id", "?")
    print(f"  multi-hop  {i+1}/{len(sample_pairs)}: {id_a} + {id_b}, {persona.name}")
    test = generator.generate_multi_hop([format_doc(a), format_doc(b)], persona)
    synthetic_tests.append({"user_input": test.question, "reference": test.answer, "query_type": "multi_hop", "persona": persona.name, "source_cves": f"{id_a}, {id_b}"})

# Global
global_sample = random.sample(documents, min(5, len(documents)))
persona = random.choice(personas)
cve_ids = ", ".join(d["metadata"].get("cve_id", "?") for d in global_sample)
print(f"  global: {cve_ids}, {persona.name}")
test = generator.generate_global([format_doc(d) for d in global_sample], persona)
synthetic_tests.append({"user_input": test.question, "reference": test.answer, "query_type": "global", "persona": persona.name, "source_cves": cve_ids})

print(f"\n{len(synthetic_tests)} synthetic tests generated")
synthetic_tests

  single-hop 1/5: CVE-2014-0160, security analyst
  single-hop 2/5: CVE-2021-45046, infrastructure engineer
  single-hop 3/5: CVE-2021-40539, product manager
  single-hop 4/5: CVE-2023-23397, security analyst
  single-hop 5/5: CVE-2022-1388, infrastructure engineer
  multi-hop  1/3: CVE-2021-41773 + CVE-2023-23397, security analyst
  multi-hop  2/3: CVE-2014-0160 + CVE-2022-3786, infrastructure engineer
  multi-hop  3/3: CVE-2021-44228 + CVE-2021-40539, product manager
  global: CVE-2021-26855, CVE-2021-41773, CVE-2021-25741, CVE-2021-44228, CVE-2022-22965, security analyst

9 synthetic tests generated


[{'user_input': 'What kind of impact does Heartbleed have?',
  'reference': 'Heartbleed allows a remote attacker to obtain sensitive information from process memory by sending crafted Heartbeat Extension packets that trigger a buffer over-read. The issue affects the TLS and DTLS implementations in OpenSSL and can expose data such as private keys. It is rated HIGH with a CVSS score of 7.5 and has a NETWORK attack vector.',
  'query_type': 'single_hop',
  'persona': 'security analyst',
  'source_cves': 'CVE-2014-0160'},
 {'user_input': 'What kind of impact can this Log4j issue have in containerized services?',
  'reference': 'This vulnerability is rated CRITICAL with a CVSS score of 9.0 and is exploitable over the network. In affected setups, an attacker who can control certain logging input can trigger an information leak and remote code execution in some environments, with local code execution possible in all environments. The documented fix is to use Log4j 2.16.0 for Java 8 or 2.12.2 

### Save to file

In [8]:
import json

manual_cases = [
    {
        "user_input": "What is the CVSS score and attack vector for Log4Shell?",
        "reference": "Log4Shell (CVE-2021-44228) has a CVSS base score of 10.0 (CRITICAL) with a NETWORK attack vector.",
    },
    {
        "user_input": "What remediation steps are recommended for Spring4Shell?",
        "reference": "Spring4Shell (CVE-2022-22965) should be remediated by upgrading Spring Framework to 5.3.18+ or 5.2.20+. Affected deployments are JDK 9+ running as WAR on Apache Tomcat.",
    },
    {
        "user_input": "What is the Heartbleed vulnerability and what does it expose?",
        "reference": "Heartbleed (CVE-2014-0160) is a buffer over-read in OpenSSL's TLS heartbeat extension that allows attackers to read up to 64KB of server memory per request, potentially exposing private keys, session tokens, and other sensitive data.",
    },
    {
        "user_input": "How does the runc container escape vulnerability work?",
        "reference": "CVE-2019-5736 allows a malicious container to overwrite the host runc binary by exploiting /proc/self/exe, gaining root-level code execution on the host.",
    },
    {
        "user_input": "Which CVEs are related to OpenSSL X.509 certificate handling?",
        "reference": "CVE-2022-3602 and CVE-2022-3786 are both buffer overflow vulnerabilities in OpenSSL's X.509 name constraint and email address verification. CVE-2022-3602 has potential RCE impact while CVE-2022-3786 leads to denial of service.",
    },
]

testset = {
    "manual": manual_cases,
    "synthetic": synthetic_tests,
}

with open("testset.json", "w") as f:
    json.dump(testset, f, indent=2)

print(f"Saved {len(manual_cases)} manual and {len(synthetic_tests)} synthetic test cases to testset.json")

Saved 5 manual and 9 synthetic test cases to testset.json
